# 💳 Credit Card Fraud Detection
> **Objective**: Identify fraudulent transactions using anomaly detection + supervised learning  
> **Dataset**: [Kaggle ULB Credit Card Fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
> **Models**: Isolation Forest · Local Outlier Factor · XGBoost  

---

## 0. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Project modules
from src.data_loader import load_dataset, get_dataset_info, download_dataset
from src.preprocessor import preprocess
from src.anomaly_detection import (
    train_isolation_forest, train_lof,
    evaluate_anomaly_model, plot_anomaly_scores
)
from src.classifier import (
    train_xgboost, evaluate_classifier,
    get_feature_importance, cross_validate_model
)
from src.visualizer import (
    plot_class_distribution, plot_roc_curve,
    plot_confusion_matrix, plot_precision_recall_curve,
    plot_feature_importance
)

print('All imports successful ✅')
print(f'NumPy: {np.__version__}  Pandas: {pd.__version__}')

## 1. Load Dataset

Place `creditcard.csv` in the `data/` folder, or use the Kaggle API:
```bash
kaggle datasets download -d mlg-ulb/creditcardfraud -p data/ --unzip
```

In [ ]:
# Uncomment to download via API:
# download_dataset()

df = load_dataset('../data/creditcard.csv')
info = get_dataset_info(df)

print('\n📊 Dataset Summary')
print('=' * 40)
for k, v in info.items():
    if k != 'features':
        print(f'  {k:25s}: {v}')

In [ ]:
df.head()

In [ ]:
df.describe().T

## 2. Exploratory Data Analysis

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')

# Bar chart
ax = axes[0]
ax.set_facecolor('#1e2130')
counts = df['Class'].value_counts()
bars = ax.bar(['Legitimate', 'Fraudulent'], counts.values,
               color=['#4ade80', '#f87171'], width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
            f'{val:,}\n({val/len(df):.2%})',
            ha='center', color='white', fontsize=10)
ax.set_title('Class Distribution', color='white', fontsize=13)
ax.tick_params(colors='#888')
for s in ax.spines.values(): s.set_edgecolor('#333')

# Transaction amount by class
ax2 = axes[1]
ax2.set_facecolor('#1e2130')
for label, color, name in [(0, '#4ade80', 'Legit'), (1, '#f87171', 'Fraud')]:
    amounts = df[df['Class'] == label]['Amount']
    ax2.hist(amounts[amounts < 500], bins=50, alpha=0.7,
             color=color, label=f'{name} (n={len(amounts):,})', density=True)
ax2.set_title('Transaction Amount Distribution (<$500)', color='white', fontsize=13)
ax2.set_xlabel('Amount ($)', color='#aaa')
ax2.set_ylabel('Density', color='#aaa')
ax2.tick_params(colors='#888')
ax2.legend(facecolor='#2a2d3e', labelcolor='white')
for s in ax2.spines.values(): s.set_edgecolor('#333')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (sampled for speed)
fig, ax = plt.subplots(figsize=(16, 12))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1e2130')

sample = df.sample(5000, random_state=42)
corr = sample.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            ax=ax, linewidths=0.3, annot=False,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', color='white', fontsize=14, pad=15)
ax.tick_params(colors='#888')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of V features for Fraud vs Legit (top 6)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.patch.set_facecolor('#0f1117')
axes = axes.flatten()

top_features = ['V4', 'V11', 'V12', 'V14', 'V17', 'V10']

for ax, feat in zip(axes, top_features):
    ax.set_facecolor('#1e2130')
    for label, color, name in [(0, '#4ade80', 'Legit'), (1, '#f87171', 'Fraud')]:
        vals = df[df['Class'] == label][feat]
        ax.hist(vals, bins=60, alpha=0.65, color=color,
                label=name, density=True)
    ax.set_title(feat, color='white', fontsize=11)
    ax.tick_params(colors='#666', labelsize=8)
    for s in ax.spines.values(): s.set_edgecolor('#333')
    ax.legend(facecolor='#2a2d3e', labelcolor='white', fontsize=8)

plt.suptitle('Feature Distributions: Fraud vs Legitimate', color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Preprocessing & SMOTE Balancing

In [ ]:
data = preprocess(df, balance_method='smote', random_state=42)

X_train     = data['X_train']
X_test      = data['X_test']
y_train     = data['y_train']
y_test      = data['y_test']
X_train_res = data['X_train_res']
y_train_res = data['y_train_res']
features    = data['feature_names']

print(f'Training set  : {X_train.shape}')
print(f'Test set      : {X_test.shape}')
print(f'Resampled train: {X_train_res.shape}')
print(f'Resampled fraud count: {y_train_res.sum():,}')

## 4. Anomaly Detection

### 4.1 Isolation Forest

In [ ]:
iso_forest = train_isolation_forest(X_train, contamination=0.001)
if_results = evaluate_anomaly_model(iso_forest, X_test, y_test, 'Isolation Forest')

### 4.2 Local Outlier Factor

In [ ]:
lof = train_lof(X_train, contamination=0.001)
lof_results = evaluate_anomaly_model(lof, X_test, y_test, 'Local Outlier Factor')

In [ ]:
# Plot anomaly score distributions
plot_anomaly_scores(
    if_results['fraud_scores'],
    lof_results['fraud_scores'],
    y_test
)
from IPython.display import Image
Image('../assets/anomaly_scores.png')

## 5. XGBoost Classifier

In [ ]:
xgb_model = train_xgboost(X_train_res, y_train_res)

In [ ]:
xgb_results = evaluate_classifier(xgb_model, X_test, y_test, threshold=0.5)

## 6. Visualizations

In [ ]:
# ROC Curve — all models
from sklearn.metrics import roc_curve, roc_auc_score

def anomaly_roc(scores, y):
    fpr, tpr, thresh = roc_curve(y, scores)
    auc = roc_auc_score(y, scores)
    return {'roc_curve': (fpr, tpr, thresh), 'roc_auc': auc}

roc_data = {
    'XGBoost':          {'roc_curve': xgb_results['roc_curve'], 'roc_auc': xgb_results['roc_auc']},
    'Isolation Forest': anomaly_roc(if_results['fraud_scores'],  y_test),
    'LOF':              anomaly_roc(lof_results['fraud_scores'], y_test),
}

plot_roc_curve(roc_data)
Image('../assets/roc_curve.png')

In [ ]:
# Confusion Matrix — XGBoost
plot_confusion_matrix(xgb_results['confusion_matrix'], 'XGBoost')
Image('../assets/confusion_matrix_xgboost.png')

In [ ]:
# Precision-Recall Curve
plot_precision_recall_curve(xgb_results['pr_curve'], xgb_results['average_precision'])
Image('../assets/precision_recall_curve.png')

In [ ]:
# Feature Importance
fi = get_feature_importance(xgb_model, features)
plot_feature_importance(fi, top_n=20)
Image('../assets/feature_importance.png')

## 7. Threshold Analysis

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_proba = xgb_results['y_proba']
thresholds = np.arange(0.1, 0.95, 0.05)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1e2130')
ax.plot(thresholds, precisions, 'o-', color='#60a5fa', label='Precision', linewidth=2)
ax.plot(thresholds, recalls,    's-', color='#f472b6', label='Recall',    linewidth=2)
ax.plot(thresholds, f1s,        '^-', color='#a78bfa', label='F1 Score',  linewidth=2)
ax.axvline(0.5, color='#fbbf24', linestyle='--', linewidth=1.5, label='Default threshold')
ax.set_xlabel('Classification Threshold', color='#aaa')
ax.set_ylabel('Score', color='#aaa')
ax.set_title('Precision / Recall / F1 vs Threshold', color='white', fontsize=13, pad=12)
ax.legend(facecolor='#2a2d3e', labelcolor='white')
ax.tick_params(colors='#888')
for s in ax.spines.values(): s.set_edgecolor('#333')
ax.grid(True, color='#2d3148', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Best F1 threshold
best_t = thresholds[np.argmax(f1s)]
print(f'Best F1 threshold: {best_t:.2f}  F1={max(f1s):.4f}')

## 8. Summary & Results

In [ ]:
print('\n' + '='*55)
print('  📊  FINAL MODEL COMPARISON')
print('='*55)
print(f'  {'Model':<25} {'ROC-AUC':>10} {'Avg Prec':>12}')
print('-'*55)
print(f'  {'XGBoost':<25} {xgb_results["roc_auc"]:>10.4f} {xgb_results["average_precision"]:>12.4f}')
print(f'  {'Isolation Forest':<25} {roc_data["Isolation Forest"]["roc_auc"]:>10.4f} {"—":>12}')
print(f'  {'Local Outlier Factor':<25} {roc_data["LOF"]["roc_auc"]:>10.4f} {"—":>12}')
print('='*55)
print('\nNext: streamlit run app.py')